In [1]:
import sys
!{sys.executable} -m pip install azure-ai-ml azure-identity --quiet

In [2]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id="85086281-da53-4cb0-a569-b42e88af9174",
    resource_group_name="ruap_movie_genre7",
    workspace_name="ruap_movie_genreeeee7"
)

print("Povezano s workspace-om:", ml_client.workspace_name)

Povezano s workspace-om: ruap_movie_genreeeee7


In [3]:
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model = Model(
    path="deployment/model_assets",
    type=AssetTypes.CUSTOM_MODEL,
    name="genre-predictor-model",
    description="Logistic Regression model za predikciju zanra filma iz postera (ResNet18 + PCA features)"
)

registered_model = ml_client.models.create_or_update(model)
print("Model registriran:", registered_model.name, "verzija:", registered_model.version)

Uploading model_assets (0.35 MBs): 100%|██████████| 348097/348097 [00:00<00:00, 2560669.55it/s]




Model registriran: genre-predictor-model verzija: 1


In [4]:
from azure.ai.ml.entities import Environment

env = Environment(
    name="genre-prediction-env",
    description="Environment za genre prediction endpoint",
    conda_file="deployment/conda.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
)

registered_env = ml_client.environments.create_or_update(env)
print("Environment registriran:", registered_env.name, "verzija:", registered_env.version)

Environment registriran: genre-prediction-env verzija: 1


In [5]:
from azure.ai.ml.entities import ManagedOnlineEndpoint
import random

endpoint_name = f"genre-predictor-{random.randint(1000,9999)}"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Endpoint za predikciju zanra filma iz postera",
    auth_mode="key"
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()
print("Endpoint kreiran:", endpoint_name)

Endpoint kreiran: genre-predictor-9655


In [7]:
deployment = ManagedOnlineDeployment(
    name="default",
    endpoint_name=endpoint_name,
    model=registered_model,
    environment=registered_env,
    code_configuration=CodeConfiguration(
        code="deployment",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS1_v2",
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment).result()
print("Deployment kreiran.")

Instance type Standard_DS1_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint genre-predictor-9655 exists


........................................................................................................................................................................................................................................................Deployment kreiran.


In [8]:
endpoint_details = ml_client.online_endpoints.get(endpoint_name)
keys = ml_client.online_endpoints.get_keys(endpoint_name)

print("Scoring URI:", endpoint_details.scoring_uri)
print("Primary key:", keys.primary_key)

Scoring URI: https://genre-predictor-9655.italynorth.inference.ml.azure.com/score
Primary key: REDACTED_KEY


In [11]:
endpoint_details.traffic = {"default": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint_details).result()

print("Traffic ažuriran.")

Traffic ažuriran.


In [13]:
from azure.ai.ml.entities import Environment

env_v2 = Environment(
    name="genre-prediction-env",
    description="Environment za genre prediction endpoint (numpy<2 fix)",
    conda_file="deployment/conda.yaml",
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04:latest"
)

registered_env_v2 = ml_client.environments.create_or_update(env_v2)
print("Environment registriran:", registered_env_v2.name, "verzija:", registered_env_v2.version)

Environment registriran: genre-prediction-env verzija: 2


In [16]:
deployment_v3 = ManagedOnlineDeployment(
    name="default",
    endpoint_name=endpoint_name,
    model=registered_model,
    environment=registered_env_v2,
    code_configuration=CodeConfiguration(
        code="deployment",
        scoring_script="score.py"
    ),
    instance_type="Standard_DS1_v2",
    instance_count=1
)

ml_client.online_deployments.begin_create_or_update(deployment_v3).result()
print("Deployment azuriran.")

Instance type Standard_DS1_v2 may be too small for compute resources. Minimum recommended compute SKU is Standard_DS3_v2 for general purpose endpoints. Learn more about SKUs here: https://learn.microsoft.com/azure/machine-learning/referencemanaged-online-endpoints-vm-sku-list
Check: endpoint genre-predictor-9655 exists
Uploading deployment (0.35 MBs): 100%|██████████| 350811/350811 [00:00<00:00, 3093780.05it/s]




...........................................Deployment azuriran.


In [17]:
import requests
import base64
import json

test_image_path = "posters/937249.jpg"

with open(test_image_path, "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

payload = {"image": image_b64}

headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer REDACTED_KEY"
}

response = requests.post(
    "https://genre-predictor-9655.italynorth.inference.ml.azure.com/score",
    headers=headers,
    data=json.dumps(payload)
)

print("Status code:", response.status_code)
print("Response:", response.text)

Status code: 200
Response: "{\"predicted_genres\": [\"Action\", \"Crime\", \"Mystery\", \"Thriller\"], \"probabilities\": {\"Action\": 0.6682861419026844, \"Adventure\": 0.28241155166969284, \"Animation\": 0.06454497735523836, \"Comedy\": 0.29043977057106546, \"Crime\": 0.8416774448366066, \"Documentary\": 0.42997203705874937, \"Drama\": 0.4974320982808147, \"Family\": 0.10626423226037367, \"Fantasy\": 0.08549809405902266, \"History\": 0.10335747161361711, \"Horror\": 0.1137049784280857, \"Music\": 0.0018792554058969595, \"Mystery\": 0.5225401729153798, \"Romance\": 0.21098942709966836, \"Science Fiction\": 0.3930856307696341, \"TV Movie\": 0.03861233424043162, \"Thriller\": 0.629652772015027, \"War\": 0.1952041590371856, \"Western\": 0.023862581593206825}}"
